# Etapa 0 — Auditoría de Problemas Manufacturados y Generación en Caché de Referencias FEM (WSL2)

Replica en `PINN_V4` los pasos iniciales de validación previa a campaña:
1. Verificación de hardware PyTorch (`DeviceManager`) y disponibilidad de **FEniCSx** en **WSL2** (`FEniCSxAdapter`).
2. Auditoría matemática automática de todos los casos manufacturados MMS ($-\nabla\cdot(k\nabla T_{\text{exact}}) = Q$ y consistencia Dirichlet en frontera dentro de `tol = 1e-4`).
3. Generación y caché de referencias FEniCSx P2 (`fem_l1.json/.npz`) en `campaign_workspace/fem_references/<case_id>/` para los casos sin solución analítica cerrada.

In [1]:
import json
import sys
import time
from pathlib import Path
import numpy as np

# Localiza la raíz PINN_V4 aunque el cuaderno se ejecute dentro de notebooks_campaign/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks_campaign":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (SRC, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Modo de ejecución:
# - 'demo_fast': subconjunto acotado de casos y pocos pasos para verificación rápida en minutos
# - 'full_campaign': reproduce todos los casos, semillas (0,1,2) y presupuestos largos de la tesis
RUN_MODE = "full_campaign"    #"demo_fast"

CAMP_WS = ROOT / "notebooks_campaign" / "campaign_workspace"
FEM_CACHE = CAMP_WS / "fem_references"
CAMP_WS.mkdir(parents=True, exist_ok=True)
FEM_CACHE.mkdir(parents=True, exist_ok=True)

# Criterios oficiales de calificación de la campaña final de tesis (2026-10-27)
STRICT_CRITERIA = {"error_Tmax_abs_K": 0.05, "nrmse_pct": 0.25, "balance_pct": 0.5}
RELAXED_CRITERIA = {k: 2.0 * v for k, v in STRICT_CRITERIA.items()}

print(f"PINN_V4 raíz: {ROOT} | RUN_MODE={RUN_MODE!r} | Workspace campaña: {CAMP_WS}")

PINN_V4 raíz: C:\usr\ths_mia_fiis\tesis_MIA_2026-2_pinn | RUN_MODE='full_campaign' | Workspace campaña: C:\usr\ths_mia_fiis\tesis_MIA_2026-2_pinn\notebooks_campaign\campaign_workspace


In [2]:
import pinn_thermal.models  # noqa: F401
from pinn_thermal.core.artifact import SolutionArtifact
from pinn_thermal.core.manifest import write_json_atomic
from pinn_thermal.core.problem import CaseCatalog
from pinn_thermal.core.trainer import PINNTrainer, TrainConfig
from pinn_thermal.fem.runner import FEniCSxAdapter

catalog = CaseCatalog(ROOT / "cases")

def load_or_solve_reference(case_id: str, problem, use_fem: bool = True):
    """Obtiene referencia analítica exacta si existe, o carga/resuelve caché FEM P2 en WSL."""
    xy, _, _ = problem.evaluation()
    exact = problem.reference(xy)
    if exact is not None and np.isfinite(exact).all():
        return np.asarray(exact, dtype=float), "analytic_reference"

    fem_dir = FEM_CACHE / case_id
    fem_json = fem_dir / "fem_l1.json"
    if fem_json.is_file() and fem_json.with_suffix(".npz").is_file():
        art = SolutionArtifact.load(fem_json)
        return art.predict(xy)["T"], f"FEM_WSL_cache:{case_id}"

    if use_fem and FEniCSxAdapter().is_available().get("available"):
        art = FEniCSxAdapter().solve(problem, output_dir=fem_dir, level=1)
        return art.predict(xy)["T"], f"FEM_WSL_L1:{case_id}"

    return None, "physics_only_no_reference"

def extract_metrics(result: dict) -> dict:
    ev = result.get("evaluation") or {}
    bal = ev.get("balance") or {}
    return {
        "error_Tmax_abs_K": ev.get("error_Tmax_abs_K"),
        "nrmse_pct": ev.get("nrmse_pct"),
        "balance_pct": bal.get("global_derived_pct"),
        "Tmax_pred_C": ev.get("Tmax_pred_C"),
        "elapsed_s": result.get("elapsed_s"),
    }
from pinn_thermal.core.device import DeviceManager
from pinn_thermal.core.fields import evaluate_field_numpy

print("Hardware PyTorch:", DeviceManager.describe())
print("Estado FEniCSx:", FEniCSxAdapter().is_available())

Hardware PyTorch: {'torch_version': '2.9.0+xpu', 'backends': {'cuda': {'available': False, 'count': 0, 'kind': 'nvidia_cuda', 'devices': []}, 'xpu': {'available': True, 'count': 1, 'devices': ['Intel(R) Graphics']}, 'npu': {'available': False, 'count': 0, 'devices': []}, 'mps': {'available': False, 'count': 0, 'devices': []}, 'dml': {'available': False, 'count': 0, 'devices': []}, 'cpu': {'available': True, 'count': 14, 'devices': ['CPU']}}, 'recommended_device': 'xpu', 'cpu_threads': 12}
Estado FEniCSx: {'available': True, 'backend': 'wsl', 'distro': 'Ubuntu', 'python': '/home/lkoc/miniforge3/envs/fenicsx/bin/python'}


## 1. Auditoría de Consistencia Física de Casos MMS (`benchmarks/mms_*`)

In [3]:
audit_mms = {}
h = 1e-5
for entry in catalog.list_entries():
    prob = catalog.load(entry["path"])
    if prob.doc.get("reference", {}).get("exact") is None:
        continue
    # Evaluar residuo PDE mediante diferencias finitas centradas sobre puntos interiores seguros
    rng = np.random.default_rng(1)
    pts = prob.sample_region_interior(prob.regions[0], 64, rng)
    k_c = prob.field_numpy("conductivity", pts[:, 0], pts[:, 1])
    k_xp = prob.field_numpy("conductivity", pts[:, 0] + h, pts[:, 1])
    k_xm = prob.field_numpy("conductivity", pts[:, 0] - h, pts[:, 1])
    k_yp = prob.field_numpy("conductivity", pts[:, 0], pts[:, 1] + h)
    k_ym = prob.field_numpy("conductivity", pts[:, 0], pts[:, 1] - h)

    t_c = prob.reference(pts)
    t_xp = prob.reference(pts + np.array([h, 0.0]))
    t_xm = prob.reference(pts - np.array([h, 0.0]))
    t_yp = prob.reference(pts + np.array([0.0, h]))
    t_ym = prob.reference(pts - np.array([0.0, h]))

    txx = (t_xp - 2.0 * t_c + t_xm) / (h ** 2)
    tyy = (t_yp - 2.0 * t_c + t_ym) / (h ** 2)
    tx = (t_xp - t_xm) / (2.0 * h)
    ty = (t_yp - t_ym) / (2.0 * h)
    kx = (k_xp - k_xm) / (2.0 * h)
    ky = (k_yp - k_ym) / (2.0 * h)
    div_q = -(k_c * (txx + tyy) + kx * tx + ky * ty)
    q_doc = prob.field_numpy("source", pts[:, 0], pts[:, 1])
    max_res = float(np.max(np.abs(div_q - q_doc)))
    scale = max(float(np.max(np.abs(q_doc))), 1.0)

    # Consistencia de bordes Dirichlet
    bc_err = 0.0
    for seg in prob.sample_boundary_regions(32, rows=1):
        bdoc = prob.doc["boundaries"][seg["side"]]
        if bdoc["type"] == "dirichlet":
            xy_b = seg["xy"]
            given = evaluate_field_numpy(bdoc["value"], xy_b[:, 0], xy_b[:, 1])
            ref = prob.reference(xy_b)
            bc_err = max(bc_err, float(np.max(np.abs(given - ref))))

    ok = (max_res <= 1e-3 * scale) and (bc_err <= 1e-4 * max(prob.T_scale, 1.0))
    audit_mms[prob.id] = {"max_residual": max_res, "bc_error": bc_err, "ok": bool(ok)}
    print(f"MMS {prob.id:20s} | ok={ok} | max_res={max_res:.3e} | bc_err={bc_err:.3e}")

write_json_atomic(CAMP_WS / "stage0_mms_audit.json", audit_mms)

MMS annulus              | ok=True | max_res=1.776e-04 | bc_err=0.000e+00
MMS mms_constant         | ok=True | max_res=3.022e-04 | bc_err=0.000e+00
MMS mms_high_contrast    | ok=True | max_res=9.989e-04 | bc_err=0.000e+00
MMS mms_interface        | ok=True | max_res=2.960e-04 | bc_err=0.000e+00
MMS mms_layered_y        | ok=True | max_res=2.369e-04 | bc_err=0.000e+00
MMS mms_robin            | ok=True | max_res=1.655e-04 | bc_err=0.000e+00
MMS mms_sine             | ok=True | max_res=3.022e-04 | bc_err=0.000e+00
MMS mms_smooth_2d        | ok=True | max_res=3.309e-04 | bc_err=0.000e+00
MMS mms_variable         | ok=True | max_res=4.648e-04 | bc_err=0.000e+00


WindowsPath('C:/usr/ths_mia_fiis/tesis_MIA_2026-2_pinn/notebooks_campaign/campaign_workspace/stage0_mms_audit.json')

## 2. Generación / Reutilización de Referencias FEM P2 en WSL2

In [4]:
# En demo_fast se resuelven 2 casos representativos; en full_campaign todo el catálogo
all_ids = [e["id"] for e in catalog.list_entries()]
target_fem_ids = ["cable_disk", "xlpe_discrete_layers"] if RUN_MODE == "demo_fast" else all_ids

fem_status = FEniCSxAdapter().is_available()
fem_manifest = {}
if fem_status.get("available"):
    adapter = FEniCSxAdapter()
    for cid in target_fem_ids:
        prob = catalog.load(cid)
        out_dir = FEM_CACHE / cid
        art = adapter.solve(prob, output_dir=out_dir, level=1, reuse_if_valid=True)
        fem_manifest[cid] = {
            "status": "ready",
            "Tmax_C": art.result.get("Tmax_C"),
            "balance_pct": art.result.get("balance_pct"),
            "dofs": art.result.get("dofs"),
        }
        print(f"FEM L1 {cid:25s} | Tmax={art.result.get('Tmax_C'):.3f} °C | balance={art.result.get('balance_pct'):.4f}%")
else:
    print("FEniCSx no disponible en este entorno:", fem_status)

write_json_atomic(CAMP_WS / "stage0_fem_cache.json", fem_manifest)

FEM L1 annulus                   | Tmax=49.567 °C | balance=0.6959%
FEM L1 cable_disk                | Tmax=138.372 °C | balance=0.0903%
FEM L1 mms_constant              | Tmax=49.996 °C | balance=0.0558%
FEM L1 mms_high_contrast         | Tmax=49.996 °C | balance=0.5304%
FEM L1 mms_interface             | Tmax=32.500 °C | balance=0.9271%
FEM L1 mms_layered_y             | Tmax=32.500 °C | balance=0.6073%
FEM L1 mms_robin                 | Tmax=35.000 °C | balance=0.0003%
FEM L1 mms_sine                  | Tmax=49.996 °C | balance=0.0558%
FEM L1 mms_smooth_2d             | Tmax=49.996 °C | balance=0.0555%
FEM L1 mms_variable              | Tmax=49.996 °C | balance=0.0531%
FEM L1 tabla_heterogenea_demo    | Tmax=62.791 °C | balance=0.5014%
FEM L1 two_layer_slab            | Tmax=98.770 °C | balance=0.0226%
FEM L1 aras_flat                 | Tmax=50.016 °C | balance=0.0029%
FEM L1 aras_single               | Tmax=63.037 °C | balance=0.0053%
FEM L1 coaxial_bc_dirichlet      | Tmax=32.144 

WindowsPath('C:/usr/ths_mia_fiis/tesis_MIA_2026-2_pinn/notebooks_campaign/campaign_workspace/stage0_fem_cache.json')